In [ ]:
# Example 1: Define a Model schema and create the table in an in-memory SQLite DB.
from peewee import SqliteDatabase, Model, CharField, IntegerField, DateField
import datetime

db = SqliteDatabase(':memory:')

class Person(Model):
    name = CharField()
    age = IntegerField()
    birthday = DateField()

    class Meta:
        database = db

db.connect()
db.create_tables([Person])

assert Person.table_exists()
assert 'name' in Person._meta.fields
print(f"Created table: {Person._meta.table_name}")
print(f"Fields: {list(Person._meta.fields.keys())}")

In [ ]:
# Example 2: Insert rows + query with .get() and .select().where().
alice = Person.create(name='Alice', age=30, birthday=datetime.date(1995, 3, 15))
bob = Person.create(name='Bob', age=25, birthday=datetime.date(2000, 7, 20))
carol = Person.create(name='Carol', age=45, birthday=datetime.date(1980, 1, 1))

fetched = Person.get(Person.name == 'Alice')
assert fetched.age == 30
assert fetched.id == alice.id

older = list(Person.select().where(Person.age > 26).order_by(Person.age))
assert len(older) == 2
assert [p.name for p in older] == ['Alice', 'Carol']

print(f"Total rows: {Person.select().count()}")
for p in Person.select():
    print(f"  {p.name} (age {p.age}, born {p.birthday})")

In [ ]:
# Example 3: ForeignKeyField + join across two related models.
from peewee import ForeignKeyField, TextField

class Note(Model):
    author = ForeignKeyField(Person, backref='notes')
    body = TextField()

    class Meta:
        database = db

db.create_tables([Note])
Note.create(author=alice, body='Hello from Alice')
Note.create(author=alice, body='Second note')
Note.create(author=bob, body='Bob was here')

alice_notes = list(alice.notes)
assert len(alice_notes) == 2

joined = (Note
          .select(Note, Person)
          .join(Person)
          .where(Person.name == 'Bob'))
joined_rows = list(joined)
assert len(joined_rows) == 1
assert joined_rows[0].author.name == 'Bob'
print(f"Alice has {len(alice_notes)} note(s); Bob's note: {joined_rows[0].body!r}")

In [ ]:
# Example 4: Update, delete, and aggregate (fn.COUNT / fn.AVG).
from peewee import fn

n_updated = Person.update(age=Person.age + 1).where(Person.name == 'Bob').execute()
assert n_updated == 1
assert Person.get(Person.name == 'Bob').age == 26

avg_age = Person.select(fn.AVG(Person.age)).scalar()
assert avg_age is not None
print(f"Average age after update: {avg_age:.2f}")

count_before = Person.select().count()
n_deleted = Person.delete().where(Person.name == 'Carol').execute()
assert n_deleted == 1
assert Person.select().count() == count_before - 1
print(f"Deleted {n_deleted} row; remaining: {Person.select().count()}")

In [ ]:
# Example 5: Transactions with atomic() — commit on success, rollback on error.
from peewee import IntegrityError

before = Person.select().count()

with db.atomic():
    Person.create(name='Dan', age=33, birthday=datetime.date(1992, 5, 5))
    Person.create(name='Eve', age=28, birthday=datetime.date(1997, 11, 11))

assert Person.select().count() == before + 2

before_rb = Person.select().count()
try:
    with db.atomic():
        Person.create(name='Frank', age=40, birthday=datetime.date(1985, 2, 2))
        raise ValueError("trigger rollback")
except ValueError:
    pass

assert Person.select().count() == before_rb, "transaction should have rolled back"
print(f"Rows after atomic+rollback: {Person.select().count()} (rollback worked)")

db.close()
assert db.is_closed()
print("peewee smoke test passed")